# Bussin -- Kaggle TPU worker

TPU v3-8 runs on a **quota separate from the 30 GPU hours** (~20 h/week, 9 h sessions) and is roughly 4x the effective throughput of T4 x2. It is what makes `bussin-1b` reachable at Chinchilla-optimal token counts -- verify your own quota on the Kaggle settings page before planning around it.

**Before running:**

1. Settings -> Accelerator -> **TPU VM v3-8**
2. Settings -> Internet -> **On**
3. Add-ons -> Secrets -> `HF_TOKEN`
4. Add Data -> `bussin-corpus`
5. **Save & Run All**

TPU sessions are 9 h, not 12 h. The watchdog reads that from `SESSION_LIMITS` automatically.


In [ ]:
# --- Bussin worker setup ---------------------------------------------
# Pinned and quiet: every second here is a second of quota not spent on
# matrix multiplies. Target is >= 95% of session wall time in training steps.
import os, subprocess, sys, time
T0 = time.time()

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"      # parallel checkpoint pulls
os.environ["TOKENIZERS_PARALLELISM"] = "false"

if not os.path.exists("bussin"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/CHANGEME/bussin.git", "."], check=False)

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "hf_transfer", "safetensors", "huggingface_hub", "pyyaml", "regex"],
               check=False)

sys.path.insert(0, os.getcwd())
print(f"setup took {time.time() - T0:.1f}s")

# torch_xla is preinstalled on Kaggle TPU VMs. Importing it here so a missing
# runtime fails now rather than 40 minutes in.
import torch_xla.core.xla_model as xm
print("XLA device:", xm.xla_device())
print("XLA world size:", xm.xrt_world_size() if hasattr(xm, "xrt_world_size") else 8)

In [ ]:
# --- Credentials ------------------------------------------------------
# Store the token in Kaggle "Add-ons -> Secrets" as HF_TOKEN. Never paste a
# token into a notebook cell: notebooks get shared, and the token grants write
# access to your checkpoint repo.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF token loaded from Kaggle secrets")
except Exception as exc:
    print(f"could not load Kaggle secret ({exc}); falling back to env HF_TOKEN")
    assert os.environ.get("HF_TOKEN"), "HF_TOKEN is required"

In [ ]:
CONFIG = "configs/400m.yaml"

In [ ]:
# --- What did we wake up on? -----------------------------------------
from bussin.relay.platform import get_platform_info, plan_batch

info = get_platform_info()
print(info)

# The bf16 question decides the whole precision path. Kaggle's T4 is Turing
# (sm75) and the P100 is Pascal (sm60); bf16 tensor cores start at Ampere, so
# on Kaggle GPUs this always prints False and training runs fp16 + GradScaler.
print(f"bf16 available: {info.supports_bf16}")
if info.device_type == "cuda" and "P100" in info.device_name:
    print("WARNING: P100 selected. It has no tensor cores and is roughly 6x "
          "slower than T4 x2 for the same quota hour. Switch to T4 x2.")

In [ ]:
# --- Verify before spending quota -------------------------------------
# A dry run does everything except train: claims and releases the lease, pulls
# the checkpoint, rebuilds the model, checks the batch plan reproduces the
# global batch exactly, and probes loss continuity. Run it first.
from bussin.relay.bootstrap import run
run(CONFIG, dry_run=True)

In [ ]:
# --- Train ------------------------------------------------------------
# The worker claims the lease, restores the run, trains until the watchdog
# fires at (session limit - 20 min), checkpoints, releases the lease, exits.
#
# If another worker holds a live lease this exits in seconds without starting,
# so a scheduled notebook costs nothing when the run is already being carried.
from bussin.relay.bootstrap import run

exit_code = run(CONFIG)
print(f"worker exited {exit_code}")